# Llama 3.1 8B · smaller-noise development experiment
**Seed 123 · standard / clipping only / σ = 0.1, 0.25, 0.5 · 63 updates · Colab L4**

Run Installation, restart the session, then run Sections 1–8 in order. Upload the downloaded `llama31_8b_low_noise_report.zip` when complete.

This exploratory follow-up tests whether the original noise range already lay beyond a transition in accuracy or uncertainty. Smaller σ means less added noise and a larger privacy budget ε. Section 3 calculates ε at δ = 1e-5 before training; no ε values are assumed.

The five conditions share fresh Poisson batches within this run and identical initial adapters. Standard and clipping-only controls are retrained, and σ = 0.5 provides a bridge to the earlier experiment. Historical runs used different random batches: compare against this run's controls first. Seed 123 controls adapter initialization; sampling and noise remain fresh research randomness.

Outputs are isolated under `checkpoints/llama31_8b_low_noise_pilot/seed123_v1`. Existing matched-pilot outputs are not overwritten. Keep the same 995 training records, five fixed demonstrations, 200 development questions, learning rate, clipping norm and training steps. No test-set evaluation is performed.

Completed suites can be loaded again. An interrupted training suite cannot resume midway: preserve it, increment RUN_VERSION, and rerun all five conditions. Development generation resumes from its cache.

This notebook produces training diagnostics and automatic development accuracy, not uncertainty results. Do not use the old uncertainty notebook unchanged: its paths and conditions target the previous experiment. Answer-label audit and clustering sensitivity analysis remain pending; this follow-up does not resolve them.


## Installation · run once, then restart session

In [ ]:
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" "accelerate==1.1.1" "huggingface-hub==0.26.2" "safetensors==0.4.5" "opacus==1.6.0" "numpy<3" "pandas<3"
# Restart session now. Then begin Section 1.

## 1 · Setup and freeze the revised cohort

In [ ]:
from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
import json, os, hashlib, gc, math, random, time, secrets, uuid, zipfile, re, string
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed, GenerationConfig, StoppingCriteria, StoppingCriteriaList
from peft import LoraConfig, get_peft_model, PeftModel, get_peft_model_state_dict
from IPython.display import display
from tqdm.auto import tqdm
from opacus.optimizers import DPOptimizer
from opacus.accountants import RDPAccountant
from opacus.accountants.analysis import rdp as rdp_analysis

drive.mount('/content/drive')
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
assert 'L4' in torch.cuda.get_device_name(0), 'Use the validated L4 configuration.'
PROJECT_DIR=Path('/content/drive/MyDrive/dp_uq_experiment')
PRE=PROJECT_DIR/'results/llama31_8b_preflight_v1'
RUN_MODE='llama_low_noise_development_pilot'
RUN_VERSION='v1'  # New version ONLY for an intentionally new attempt.
SEED=789
ROOT=PROJECT_DIR/'checkpoints/llama31_8b_low_noise_pilot'/f'seed{SEED}_{RUN_VERSION}'
CLIP_NORM=1.0; EXPECTED_BATCH_SIZE=16; NOISE_MULTIPLIERS=(0.1,0.25,0.5)
DELTA=1e-5; STEPS=63; MAX_LENGTH=256

def read_json(p):return json.loads(p.read_text())
def save_json(v,p):
    temp=p.with_suffix(p.suffix+'.tmp')
    temp.write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False));os.replace(temp,p)
def fingerprint(v):return hashlib.sha256(json.dumps(v,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def file_hash(p):return hashlib.sha256(p.read_bytes()).hexdigest()
pre=read_json(PRE/'manifest.json')
assert read_json(PRE/'summary.json')['n']==200
MODEL_ID=pre['model_id'];MODEL_REVISION=pre['revision']
assert MODEL_ID=='meta-llama/Llama-3.1-8B'
shots=read_json(PRE/'demonstrations.json');assert shots==pre['shots'] and len(shots)==5
all_training=read_json(PROJECT_DIR/'data/qa_train1000_seed123_lora_v1.json')['records']
splits=read_json(PROJECT_DIR/'data/qa_eval_splits_v1.json')
assert fingerprint(all_training)==pre['train_hash'] and fingerprint(splits)==pre['split_hash']
shot_ids={r['question_id'] for r in shots}
assert len(shot_ids)==5 and all(s in all_training for s in shots)
selected=[r for r in all_training if r['question_id'] not in shot_ids]
assert len(selected)==995 and len({r['question_id'] for r in selected})==995
N=len(selected); SAMPLE_RATE=EXPECTED_BATCH_SIZE/N
assert 0<DELTA<1/N
prior={'lora':{'r':8,'alpha':16,'targets':['q_proj','v_proj'],'dropout':0.0}}
versions={p:metadata.version(p) for p in ['torch','transformers','peft','accelerate','opacus','numpy','tokenizers','huggingface-hub','safetensors']}
assert versions['opacus']=='1.6.0'
try:HF_TOKEN=userdata.get('HF_TOKEN')
except Exception:HF_TOKEN=getpass('Hugging Face token (hidden): ')
assert HF_TOKEN
print('Protected selected records:',N,'| Fixed public demonstrations:',len(shots))
print('Poisson rate:',SAMPLE_RATE,'| Output:',ROOT)


## 2 · Prompt, answer-only loss and experiment identity
The five-shot prompt and greedy evaluation boundary match the preflight. Training targets include a leading space, the short reference answer and EOS. Prompt tokens are ignored by the loss. All 995 examples must fit 256 tokens; stop rather than silently dropping any. Public demonstrations, protected records and held-out questions must be disjoint by ID and normalized text.

In [ ]:
import re, unicodedata

def question_key(text):
    return re.sub(r'\W+',' ',unicodedata.normalize('NFKC',text).casefold()).strip()
heldout = splits['dev']+splits['test']
assert not ({r['question_id'] for r in selected}&{r['question_id'] for r in heldout})
assert not ({question_key(r['question']) for r in selected}&{question_key(r['question']) for r in heldout})
assert not ({question_key(r['question']) for r in selected}&{question_key(r['question']) for r in shots})
assert not ({r['question_id'] for r in shots}&{r['question_id'] for r in heldout})
assert not ({question_key(r['question']) for r in shots}&{question_key(r['question']) for r in heldout})
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=HF_TOKEN)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
assert tokenizer.bos_token_id is not None and tokenizer.eos_token_id is not None

def prompt_ids(question):
    examples='\n\n'.join('Question: '+r['question']+'\nAnswer: '+r['answer'].strip() for r in shots)
    prompt=examples+'\n\nQuestion: '+question+'\nAnswer:'
    return [tokenizer.bos_token_id]+tokenizer.encode(prompt,add_special_tokens=False)

def encode(row):
    p=prompt_ids(row['question'])
    a=tokenizer(' '+row['answer'].strip(),add_special_tokens=False)['input_ids']
    assert a
    a=a+[tokenizer.eos_token_id]
    assert len(p)+len(a)<=MAX_LENGTH
    return {'input_ids':p+a,'labels':[-100]*len(p)+a}

encoded=[encode(r) for r in selected]

CONDITIONS = [
    {'condition':'standard_poisson','clip_norm':None,'noise_multiplier':0.0},
    {'condition':'clipping_poisson','clip_norm':CLIP_NORM,'noise_multiplier':0.0},
    {'condition':'dp_sigma_0p1','clip_norm':CLIP_NORM,'noise_multiplier':0.1},
    {'condition':'dp_sigma_0p25','clip_norm':CLIP_NORM,'noise_multiplier':0.25},
    {'condition':'dp_sigma_0p5','clip_norm':CLIP_NORM,'noise_multiplier':0.5},
]

COMMON = {
    'engine':'explicit_per_example_opacus_poisson_v1', 'run_mode':RUN_MODE,
    'initialization_seed':SEED, 'model_id':MODEL_ID,'model_revision':MODEL_REVISION,
    'training_data_hash':fingerprint(selected),'evaluation_split_hash':fingerprint(splits),
    'nominal_training_records':N,'sample_rate':SAMPLE_RATE,
    'expected_batch_size':EXPECTED_BATCH_SIZE,'steps':STEPS,'max_length':MAX_LENGTH,
    'sampling':'independent Bernoulli inclusion per record per logical step; shared in-memory draws',
    'normalization':'fixed expected batch size including empty draws',
    'base_dtype':'bfloat16','adapter_dtype':'float32','lora':prior['lora'],
    'learning_rate':1e-4,'betas':[0.9,0.999],'adam_epsilon':1e-8,
    'weight_decay':0.0,'scheduler':'constant',
    'loss':'mean answer+EOS CE per record; gradient sum divided by expected batch size',
    'accountant':'rdp','delta':DELTA,'adjacency':'add_remove_one_selected_QA_record',
    'secure_mode':False,'production_private_release':False,
    'public_demonstrations':shots,'public_demonstration_hash':fingerprint(shots),
    'preflight_manifest_hash':fingerprint(pre),'original_train_hash':fingerprint(all_training),
    'protected_scope':'995 selected records; conditional on fixed public demonstrations and selected cohort',
    'prompt_version':'five_shot_qa_completion_newline_v1','gradient_checkpointing':'non-reentrant',
    'randomness':'fresh independent research PRNGs; sampling and noise seeds not persisted',
    'versions':versions,'gpu':torch.cuda.get_device_name(0),
}
CONFIG = {'common':COMMON,'conditions':CONDITIONS}
print('Expected presentations per condition:',STEPS * EXPECTED_BATCH_SIZE)
print('Poisson inclusion rate:',SAMPLE_RATE)

## 3 · Verify accumulation, clipping, empty draws and accounting
These tests compare streamed per-record gradients with dense Opacus updates at all three noise levels. They check accumulator reset, fixed expected-batch normalization, Gaussian scaling on empty draws, and consistency between two RDP accounting interfaces. All must pass before training.

In [ ]:
def joint_norm(tensors):
    return torch.sqrt(sum(t.detach().double().square().sum() for t in tensors))

def deliver_samples(optimizer, params, gradients, final_microbatch):
    # gradients are already individual-example gradients, never a batch average.
    for p,g in zip(params,gradients):
        p.grad_sample = g.detach().unsqueeze(0)
    optimizer.signal_skip_step(do_skip=not final_microbatch)
    optimizer.step()

def deliver_empty(optimizer, params):
    optimizer.zero_grad(set_to_none=True)
    for p in params:
        p.grad_sample = torch.empty((0, *p.shape), device=p.device, dtype=p.dtype)
    optimizer.signal_skip_step(do_skip=False)
    optimizer.step()

def toy_update(streamed, sigma, examples, denominator=16):
    params=[torch.nn.Parameter(torch.zeros(1)),torch.nn.Parameter(torch.zeros(1))]
    opt=DPOptimizer(torch.optim.SGD(params,lr=1.0),noise_multiplier=sigma,
        max_grad_norm=1.0,expected_batch_size=denominator,
        generator=torch.Generator().manual_seed(719),secure_mode=False)
    if not examples:
        deliver_empty(opt,params)
    elif streamed:
        for j,example in enumerate(examples):
            opt.zero_grad(set_to_none=True)
            deliver_samples(opt,params,[torch.tensor([x]) for x in example],j==len(examples)-1)
    else:
        opt.zero_grad(set_to_none=True)
        for i,p in enumerate(params):p.grad_sample=torch.tensor([[e[i]] for e in examples])
        opt.step()
    return torch.cat([p.detach() for p in params])

toy_examples=[[6.0,8.0],[0.0,1.0]]
for sigma in [0.0,*NOISE_MULTIPLIERS]:
    for examples in [toy_examples,[],[[0.1,0.2]]]:
        assert torch.allclose(toy_update(True,sigma,examples),toy_update(False,sigma,examples),atol=1e-7)
# Opacus uses norm + 1e-6; allow the tiny resulting difference from exact clipping.
assert torch.allclose(toy_update(True,0.0,toy_examples),-torch.tensor([0.6,1.8])/16,atol=1e-6)
assert torch.equal(toy_update(True,0.0,[]),torch.zeros(2))
assert not torch.equal(toy_update(True,1.0,[]),torch.zeros(2))
# An empty step contains only Gaussian noise; replay a *test-only* seed to verify scaling.
g=torch.Generator().manual_seed(719)
expected=-torch.cat([torch.normal(0.,1.,size=(1,),generator=g) for _ in range(2)])/16
assert torch.allclose(toy_update(True,1.0,[]),expected,atol=1e-7)
# Check accumulator reset across consecutive updates, including an empty middle update.
p=torch.nn.Parameter(torch.zeros(2))
opt=DPOptimizer(torch.optim.SGD([p],lr=1.0),noise_multiplier=0.,max_grad_norm=1.,expected_batch_size=16)
for examples in [toy_examples,[],toy_examples]:
    if not examples:deliver_empty(opt,[p])
    for j,e in enumerate(examples):
        opt.zero_grad(set_to_none=True)
        deliver_samples(opt,[p],[torch.tensor(e)],j==len(examples)-1)
assert torch.allclose(p.detach(),-2*torch.tensor([0.6,1.8])/16,atol=1e-6)

def accountant_for(steps, sigma):
    a=RDPAccountant()
    for _ in range(steps): a.step(noise_multiplier=sigma,sample_rate=SAMPLE_RATE)
    return a

PLANNED_EPSILON = {}
for sigma in NOISE_MULTIPLIERS:
    a=accountant_for(STEPS,sigma)
    epsilon=a.get_epsilon(delta=DELTA)
    orders=RDPAccountant.DEFAULT_ALPHAS
    direct=rdp_analysis.compute_rdp(q=SAMPLE_RATE,noise_multiplier=sigma,steps=STEPS,orders=orders)
    direct_epsilon,_=rdp_analysis.get_privacy_spent(orders=orders,rdp=direct,delta=DELTA)
    assert math.isfinite(epsilon) and epsilon>0
    assert abs(epsilon-direct_epsilon)<1e-10
    assert accountant_for(STEPS+1,sigma).get_epsilon(delta=DELTA)>=epsilon
    assert sum(h[2] for h in a.history)==STEPS
    PLANNED_EPSILON[sigma]=epsilon
assert PLANNED_EPSILON[0.1]>PLANNED_EPSILON[0.25]>PLANNED_EPSILON[0.5]
print('PASS: streamed/dense updates at all noise levels, clipping, normalization, empty batches, reset, accounting.')
display(pd.DataFrame([{'noise_multiplier':s,'planned_standalone_epsilon':e,'delta':DELTA,'steps':STEPS}
                      for s,e in PLANNED_EPSILON.items()]))
MECHANISM_CHECKS_PASSED=True


## 4 · Training functions
Each condition loads the pinned base, initializes identical FP32 LoRA weights, and processes one record at a time. Per-example gradients are passed explicitly to Opacus; no batch-averaged gradient is treated as an individual gradient. Empty Poisson draws still make a logical update and receive noise/accounting. Research noise RNGs are independent of the public initialization seed.

In [ ]:
def adapter_state(model):
    return {k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}

def state_hash(state):
    h=hashlib.sha256()
    for k in sorted(state):
        t=state[k].contiguous()
        h.update(k.encode());h.update(str(tuple(t.shape)).encode());h.update(str(t.dtype).encode())
        h.update(t.view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

def new_model():
    set_seed(SEED)
    model=AutoModelForCausalLM.from_pretrained(
        MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.bfloat16,
        attn_implementation='sdpa',token=HF_TOKEN,device_map={'':'cuda:0'},low_cpu_mem_usage=True)
    model=get_peft_model(model,LoraConfig(task_type='CAUSAL_LM',r=8,lora_alpha=16,
                   target_modules=['q_proj','v_proj'],lora_dropout=0.0,bias='none'))
    for p in model.parameters():
        if p.requires_grad: p.data=p.data.float()
    assert all('lora_' in name for name,p in model.named_parameters() if p.requires_grad)
    assert not any(float(m.p)>0 for m in model.modules() if isinstance(m,torch.nn.Dropout))
    model.config.use_cache=False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    model.train()
    model.print_trainable_parameters()
    return model

def example_loss(model, row):
    ids=torch.tensor([row['input_ids']],device='cuda',dtype=torch.long)
    labels=torch.tensor([row['labels']],device='cuda',dtype=torch.long)
    with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
        logits=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False).logits
        targets=labels[:,1:]
        assert targets.ne(-100).any()
        loss=F.cross_entropy(logits[:,:-1,:].float().reshape(-1,logits.shape[-1]),
                             targets.reshape(-1),ignore_index=-100,reduction='mean')
    if not torch.isfinite(loss): raise FloatingPointError('Non-finite example loss.')
    return loss


def build_optimizer(params, spec, noise_generator):
    base=torch.optim.AdamW(params,lr=COMMON['learning_rate'],betas=tuple(COMMON['betas']),
        eps=COMMON['adam_epsilon'],weight_decay=0.0,foreach=False)
    if spec['clip_norm'] is None:return base
    return DPOptimizer(base,noise_multiplier=spec['noise_multiplier'],
        max_grad_norm=spec['clip_norm'],expected_batch_size=EXPECTED_BATCH_SIZE,
        generator=noise_generator,secure_mode=False)

def train_condition(spec, batches, session_id, expected_initial_hash=None):
    condition=spec['condition']; is_dp=spec['noise_multiplier']>0
    planned_epsilon=PLANNED_EPSILON[spec['noise_multiplier']] if is_dp else None
    run_dir=ROOT/condition
    run_dir.mkdir(parents=True,exist_ok=False)
    cfg={**COMMON,**spec,'matched_session_id':session_id,
         'mechanism':'accounted_noisy_clipped_training' if is_dp else 'non_private_control'}
    save_json(cfg,run_dir/'run_manifest.json')
    model=new_model()
    params=[p for p in model.parameters() if p.requires_grad]
    initial_hash=state_hash(adapter_state(model))
    if expected_initial_hash is not None:assert initial_hash==expected_initial_hash
    # Independent noise generator: never reset to the public initialization seed.
    noise_generator=torch.Generator(device='cuda')
    noise_generator.manual_seed(secrets.randbits(63))
    optimizer=build_optimizer(params,spec,noise_generator)
    accountant=RDPAccountant() if is_dp else None
    started=time.time();torch.cuda.reset_peak_memory_stats()
    logs=[];loss_total=0.;presentations=0;clipped=0;noise_updates=0
    standard_sums=[torch.zeros_like(p) for p in params] if spec['clip_norm'] is None else None
    for step,batch in enumerate(batches,1):
        if standard_sums is not None:
            for total in standard_sums:total.zero_()
        step_loss=0.;step_clipped=0
        if len(batch)==0 and spec['clip_norm'] is not None:
            deliver_empty(optimizer,params)
        for j,index in enumerate(batch):
            optimizer.zero_grad(set_to_none=True)
            loss=example_loss(model,encoded[int(index)])
            if not is_dp:step_loss+=float(loss.detach())
            loss.backward();del loss
            if any(p.grad is None for p in params):raise RuntimeError('Missing adapter gradient.')
            grads=[p.grad for p in params]
            norm=float(joint_norm(grads))
            if not math.isfinite(norm):raise FloatingPointError('Non-finite record gradient; run incomplete.')
            if not is_dp and spec['clip_norm'] is not None:step_clipped+=int(norm>spec['clip_norm'])
            if standard_sums is not None:
                with torch.no_grad():
                    for total,grad in zip(standard_sums,grads):total.add_(grad)
            else:
                deliver_samples(optimizer,params,grads,j==len(batch)-1)
            del grads
        if standard_sums is not None:
            optimizer.zero_grad(set_to_none=True)
            for p,total in zip(params,standard_sums):p.grad=total/EXPECTED_BATCH_SIZE
            optimizer.step()
        # Only one accountant event per complete Poisson draw (including empty draws).
        if is_dp:
            accountant.step(noise_multiplier=spec['noise_multiplier'],sample_rate=SAMPLE_RATE)
            noise_updates+=1
        if not all(bool(torch.isfinite(p).all()) for p in params):
            raise FloatingPointError('Non-finite adapter weights; run incomplete.')
        underlying=optimizer.original_optimizer if isinstance(optimizer,DPOptimizer) else optimizer
        assert all(int(underlying.state[p]['step'])==step for p in params), 'Unexpected number of AdamW updates.'
        entry={'step':step,'accounted_epsilon':accountant.get_epsilon(delta=DELTA) if is_dp else None}
        if not is_dp:
            presentations+=len(batch);loss_total+=step_loss;clipped+=step_clipped
            entry.update({'batch_size':len(batch),'mean_example_loss':step_loss/len(batch) if len(batch) else None,
                          'fraction_clipped':step_clipped/len(batch) if len(batch) else None})
        logs.append(entry)
        save_json({'completed_steps':step,'status':'training'},run_dir/'progress.json')
        if step%5==0 or step==STEPS:print(condition,entry)
    assert len(logs)==STEPS
    if is_dp:
        assert noise_updates==STEPS and sum(h[2] for h in accountant.history)==STEPS
        assert abs(accountant.get_epsilon(delta=DELTA)-planned_epsilon)<1e-10
        save_json({'accountant':'rdp','history':accountant.history,'epsilon':planned_epsilon,
            'delta':DELTA,'sample_rate':SAMPLE_RATE,'steps':STEPS,
            'noise_multiplier':spec['noise_multiplier'],'scope':'standalone final noisy adapter; research RNG'},
            run_dir/'privacy_accounting.json')
    final_hash=state_hash(adapter_state(model))
    assert final_hash!=initial_hash,'Adapter did not change.'
    model.save_pretrained(run_dir/'adapter');tokenizer.save_pretrained(run_dir/'adapter')
    save_json(logs,run_dir/'step_log.json')
    pd.DataFrame(logs).to_csv(run_dir/'step_log.csv',index=False)
    peak=torch.cuda.max_memory_allocated()/1024**3
    # Clear remaining tensor references, including loop variables, before reload.
    optimizer.zero_grad(set_to_none=True)
    del optimizer,underlying,params,standard_sums,model,noise_generator
    if 'p' in locals():del p
    if 'total' in locals():del total
    if 'grad' in locals():del grad
    gc.collect();torch.cuda.empty_cache()
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,
        torch_dtype=torch.bfloat16,attn_implementation='sdpa',token=HF_TOKEN,device_map={'':'cuda:0'},low_cpu_mem_usage=True)
    reloaded=PeftModel.from_pretrained(base,run_dir/'adapter',is_trainable=False).eval()
    assert state_hash(adapter_state(reloaded))==final_hash,'Adapter reload mismatch.'
    del reloaded,base
    gc.collect();torch.cuda.empty_cache()
    summary={'initialization_seed':SEED,'condition':condition,'run_mode':RUN_MODE,'matched_session_id':session_id,
        'optimizer_steps':STEPS,'expected_presentations':STEPS*EXPECTED_BATCH_SIZE,
        'sample_rate':SAMPLE_RATE,'expected_batch_size':EXPECTED_BATCH_SIZE,
        'clip_norm':spec['clip_norm'],'noise_multiplier':spec['noise_multiplier'],
        'accounted_epsilon':planned_epsilon if is_dp else None,'delta':DELTA if is_dp else None,
        'mean_example_loss':None if is_dp else loss_total/presentations if presentations else None,
        'fraction_clipped':None if is_dp else clipped/presentations if presentations else None,
        'peak_gpu_gb':peak,'elapsed_seconds':time.time()-started,
        'initial_adapter_hash':initial_hash,'final_adapter_hash':final_hash,
        'adapter_reload_verified':True,'all_finite_checks_passed':True,
        'adapter_path':str(run_dir/'adapter'),'secure_mode':False,
        'status':'accounted DP mechanism; research implementation' if is_dp else 'non-private control'}
    save_json(summary,run_dir/'summary.json')
    return summary

## 5 · Train the matched five-condition suite
Leave this running until all five finish. Adapter files stay in Drive. Completed suites are reused on rerun; incomplete suites require a new version and full matched restart. No sampling/noise RNG states are exported. The old smoke adapters are not loaded.

In [ ]:
def run_suite():
    assert MECHANISM_CHECKS_PASSED
    if (ROOT/'suite_summary.json').exists():
        assert read_json(ROOT/'suite_config.json')==CONFIG,'Settings changed: choose a new RUN_VERSION.'
        summaries=read_json(ROOT/'suite_summary.json')
        assert len(summaries)==len(CONDITIONS)
        assert {s['condition'] for s in summaries}=={s['condition'] for s in CONDITIONS}
        assert all(s['optimizer_steps']==STEPS and s['adapter_reload_verified'] and s['all_finite_checks_passed'] for s in summaries)
        assert len({s['initial_adapter_hash'] for s in summaries})==1
        assert len({s['matched_session_id'] for s in summaries})==1
        for s in summaries:
            assert (Path(s['adapter_path'])/'adapter_model.safetensors').exists()
        print('Loaded completed matched suite; no retraining.')
        return summaries
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError('Incomplete suite exists. Keep its files, change RUN_VERSION, and rerun all five conditions.')
    ROOT.mkdir(parents=True,exist_ok=True)
    save_json(CONFIG,ROOT/'suite_config.json')
    save_json({'sigma_to_epsilon':{str(k):v for k,v in PLANNED_EPSILON.items()},'delta':DELTA},ROOT/'planned_privacy.json')
    session_id=uuid.uuid4().hex # identifier only; never an RNG seed
    sampler=np.random.default_rng(secrets.randbits(128))
    batches=[np.flatnonzero(sampler.random(N)<SAMPLE_RATE) for _ in range(STEPS)]
    assert all(len(b)==len(set(b.tolist())) for b in batches)
    summaries=[]
    for spec in CONDITIONS:
        initial_hash=summaries[0]['initial_adapter_hash'] if summaries else None
        summaries.append(train_condition(spec,batches,session_id,initial_hash))
    assert len({s['initial_adapter_hash'] for s in summaries})==1
    assert len({s['matched_session_id'] for s in summaries})==1
    assert all(s['optimizer_steps']==STEPS and s['adapter_reload_verified'] for s in summaries)
    save_json(summaries,ROOT/'suite_summary.json')
    pd.DataFrame(summaries).to_csv(ROOT/'comparison_summary.csv',index=False)
    del batches,sampler
    return summaries

summaries=run_suite()
display(pd.DataFrame(summaries)[['condition','optimizer_steps','clip_norm','noise_multiplier',
    'accounted_epsilon','delta','mean_example_loss','fraction_clipped','peak_gpu_gb','adapter_reload_verified']])
print('Summary:',ROOT/'comparison_summary.csv')
for spec in CONDITIONS:
    if spec['noise_multiplier']>0:
        print('Accountant:', ROOT/spec['condition']/'privacy_accounting.json')
print('Completed seed:',SEED)
print('Continue to Section 6: development evaluation. Do not start other seeds yet.')


## 6 · Generate 200 development answers per condition
This stage checks whether fine-tuning preserves useful accuracy. It uses exactly the preflight prompt, EOS settings, 32-token cap, greedy decoding and first-newline extraction. Raw continuations remain saved. No uncertainty estimates or final-test answers are computed. Saved adapter hashes bind each cache. Empty/truncated answers are recorded, never silently replaced.

After a completed training suite, a disconnect is recoverable: rerun Sections 1–6 with the same version. Training is skipped, and generation resumes. If CUDA memory is exhausted after an error, restart the runtime first.

In [ ]:
def normalize_answer(text):
    text=text.lower().replace('_',' ')
    text=''.join(c for c in text if c not in string.punctuation)
    return ' '.join(re.sub(r'\b(a|an|the)\b',' ',text).split())
def aliases_for(item):return list(dict.fromkeys([item['reference_answer'],*item['accepted_answers']]))
def em(text,item):
    value=normalize_answer(text)
    return int(bool(value) and any(value==normalize_answer(a) for a in aliases_for(item)))
class StopAtNewline(StoppingCriteria):
    def __init__(self,start):self.start=start
    def __call__(self,input_ids,scores,**kwargs):
        return torch.tensor(['\n' in tokenizer.decode(input_ids[0,self.start:],skip_special_tokens=True)],device=input_ids.device)

DEV=splits['dev']; assert len(DEV)==200
assert fingerprint(DEV)==pre['dev_hash']
EVAL_DIR=ROOT/'development';EVAL_DIR.mkdir(exist_ok=True)
pre_cache=read_json(PRE/'generations.json')
assert pre_cache['binding']==fingerprint(pre)
assert len(pre_cache['records'])==200
assert {r['question_id'] for r in pre_cache['records']}=={r['question_id'] for r in DEV}
for r in pre_cache['records']:
    original=next(d for d in DEV if d['question_id']==r['question_id'])
    assert r['source_hash']==fingerprint(original)

@torch.inference_mode()
def generate_dev(model,item,generation_config,eos_ids):
    ids=prompt_ids(item['question']);x=torch.tensor([ids],device='cuda')
    generated=model.generate(input_ids=x,attention_mask=torch.ones_like(x),generation_config=generation_config,
        stopping_criteria=StoppingCriteriaList([StopAtNewline(len(ids))]))[0,len(ids):].tolist()
    raw=tokenizer.decode(generated,skip_special_tokens=True)
    answer=raw.split('\n',1)[0].strip()
    ended=bool(generated) and generated[-1] in eos_ids
    return {'question_id':item['question_id'],'question':item['question'],'reference_answer':item['reference_answer'],
      'accepted_answers':aliases_for(item),'source_hash':fingerprint(item),'prediction':answer,'raw_continuation':raw,
      'generated_token_ids':generated,'exact_match':em(answer,item),'raw_exact_match':em(raw.strip(),item),
      'empty_answer':not bool(answer),'hit_length_limit':len(generated)>=32 and not ended and '\n' not in raw,
      'ended_with_eos':ended}

def evaluate_condition(summary):
    name=summary['condition']; path=EVAL_DIR/(name+'_generations.json')
    adapter=Path(summary['adapter_path'])
    binding={'model_revision':MODEL_REVISION,'adapter_file_hash':file_hash(adapter/'adapter_model.safetensors'),
             'adapter_config_hash':file_hash(adapter/'adapter_config.json'),'suite_config_hash':fingerprint(CONFIG),
             'dev_hash':fingerprint(DEV),'preflight_manifest_hash':fingerprint(pre),
             'decoding':'greedy_32_tokens_first_newline'}
    cache=read_json(path) if path.exists() else {'binding':binding,'records':[]}
    assert cache['binding']==binding
    by_id={r['question_id']:r for r in cache['records']}
    assert len(by_id)==len(cache['records']) and set(by_id)<={r['question_id'] for r in DEV}
    for item in DEV:
        if item['question_id'] in by_id:assert by_id[item['question_id']]['source_hash']==fingerprint(item)
    if len(by_id)==len(DEV):return cache['records']
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,token=HF_TOKEN,
        torch_dtype=torch.bfloat16,device_map={'':'cuda:0'},attn_implementation='sdpa',low_cpu_mem_usage=True)
    model=PeftModel.from_pretrained(base,adapter,is_trainable=False).eval()
    assert state_hash(adapter_state(model))==summary['final_adapter_hash']
    model.config.use_cache=True
    eos_ids=model.generation_config.eos_token_id or tokenizer.eos_token_id
    if isinstance(eos_ids,int):eos_ids=[eos_ids]
    assert eos_ids==read_json(PRE/'tokenizer_identity.json')['eos_ids']
    config=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=32,
      eos_token_id=eos_ids,pad_token_id=tokenizer.pad_token_id,bos_token_id=tokenizer.bos_token_id,
      repetition_penalty=1.0,use_cache=True)
    for item in tqdm(DEV,desc=name):
        if item['question_id'] not in by_id:
            row=generate_dev(model,item,config,eos_ids)
            cache['records'].append(row);by_id[item['question_id']]=row;save_json(cache,path)
    del model,base;gc.collect();torch.cuda.empty_cache()
    return cache['records']

all_dev={'unmodified_base':pre_cache['records']}
for summary in summaries:all_dev[summary['condition']]=evaluate_condition(summary)
print('All five conditions have complete development outputs.')


## 7 · Summarize automatic EM and export review files
The comparison uses the same 200 questions and the same prompt/extraction policy for the unmodified model and all five adapters. Automatic alias-match is not adjudicated correctness. The previous seven development holds are not silently assumed: import their documented decisions into this review if available, then calculate eligible-cohort results later. Do not select models using prior test outcomes.

In [ ]:
table=[];review_rows=[]
for name,records in all_dev.items():
    assert len(records)==200
    table.append({'condition':name,'n':len(records),'automatic_exact_match':sum(r['exact_match'] for r in records)/len(records),
        'empty_answers':sum(r['empty_answer'] for r in records),'length_limit_hits':sum(r['hit_length_limit'] for r in records),
        'grading':'automatic_EM_all200_not_human_verified'})
    for r in records:
        review_rows.append({'review_id':fingerprint([name,r['question_id'],r['prediction']]),'condition':name,
            'question_id':r['question_id'],'question':r['question'],'prediction':r['prediction'],
            'reference_answer':r['reference_answer'],'accepted_answers':json.dumps(r['accepted_answers'],ensure_ascii=False),
            'automatic_exact_match':r['exact_match'],'raw_continuation':r['raw_continuation']})
report_table=pd.DataFrame(table)
report_table.to_csv(EVAL_DIR/'development_accuracy_summary.csv',index=False)
save_json(table,EVAL_DIR/'development_accuracy_summary.json')
display(report_table)
review=pd.DataFrame(review_rows);private=review[['review_id','condition']].copy()
private.to_csv(EVAL_DIR/'review_mapping_private.csv',index=False)
review=review.drop(columns='condition').sort_values('review_id').reset_index(drop=True)
p=EVAL_DIR/'development_answer_review.csv'
if p.exists():
    previous=pd.read_csv(p,keep_default_na=False,dtype=str)
    assert previous.review_id.is_unique and set(previous.review_id)==set(review.review_id)
    previous=previous.set_index('review_id').loc[review.review_id]
    assert previous.prediction.tolist()==review.prediction.tolist()
    for col in ['human_correct','human_reviewer','human_note']:review[col]=previous[col].tolist()
else:
    for col in ['human_correct','human_reviewer','human_note']:review[col]=''
review.to_csv(p,index=False)
validity=EVAL_DIR/'development_question_validity.csv'
if not validity.exists():
    data=pd.DataFrame(DEV)[['question_id','question','reference_answer']]
    for col in ['decision','reason','reviewer']:data[col]=''
    data.to_csv(validity,index=False)
print('Reviewer file:',p)
print('Keep private mapping and condition summaries separate when requesting blinded correctness review.')


## 8 · Export report and stop
Send `llama31_8b_low_noise_report.zip`. This analyst report includes training/accounting summaries, automatic accuracy, generated answers and a condition-free review CSV, but no adapters or credentials. For a blinded reviewer, send only the review CSV and question-validity file, not the full ZIP.

Next decision: verify development correctness and inspect accuracy changes before spending time on three seeds and ten-sample semantic entropy. This notebook does not make RQ1/RQ3 conclusions from automatic accuracy alone.

Sources: [Opacus DPOptimizer](https://opacus.ai/api/optim/dp_optimizer.html), [Opacus RDP accountant](https://opacus.ai/api/accountants/rdp.html), [Llama model card](https://huggingface.co/meta-llama/Llama-3.1-8B).

In [ ]:
assert (ROOT/'suite_summary.json').exists()
assert (EVAL_DIR/'development_accuracy_summary.json').exists()
report=ROOT/'llama31_8b_low_noise_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['suite_config.json','suite_summary.json','comparison_summary.csv','planned_privacy.json']:
        z.write(ROOT/name,name)
    for spec in CONDITIONS:
        folder=ROOT/spec['condition']
        for name in ['run_manifest.json','summary.json','privacy_accounting.json','step_log.json']:
            p=folder/name
            if p.exists():z.write(p,str(p.relative_to(ROOT)))
    for p in EVAL_DIR.iterdir():
        if p.suffix in ['.json','.csv'] and p.name!='review_mapping_private.csv':z.write(p,str(p.relative_to(ROOT)))
    z.writestr('README.txt','Single-seed development pilot. 995 protected selected records plus five fixed public demonstrations. Per-run RDP mechanism accounting, not a joint private release. Automatic EM on all 200 dev questions; human correctness and question validity review pending. No final-test inference. Analyst ZIP is not a blinded reviewer bundle. Original unmodified-model accuracy is reused from the frozen preflight.')
print('Send:',report)
from google.colab import files
files.download(str(report))
